<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/07_callbacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 07 — Callbacks as Middleware

Six lifecycle hooks wrap every invocation of your agent. Intercept any step — the agent starting, the LLM call, a tool call — observe it, transform it, or **short-circuit it entirely**. This is the cleanest mechanism in the framework for guardrails, caching, PII redaction, and cross-cutting concerns.

If you've used Express.js middleware, Django middleware, or plugin hooks in any framework, the shape is familiar. **Callbacks are for agents what middleware is for web handlers.**

The six hooks, paired before/after around three lifecycle events:

| Event | Before | After |
|---|---|---|
| Agent runs | `before_agent_callback` | `after_agent_callback` |
| Model call | `before_model_callback` | `after_model_callback` |
| Tool call | `before_tool_callback` | `after_tool_callback` |

Plus two lesser-used ones: `on_model_error_callback` and `on_tool_error_callback` for error recovery.

**The return-to-override pattern:** return `None` from a callback and the pipeline proceeds normally. Return a real object (an `LlmResponse`, a tool-return dict, etc.) and ADK uses *your value* in place of actually running the LLM or tool. That single convention is what makes callbacks a guardrail mechanism, not just an observability hook.

**What you'll build:**
- A `before_model_callback` blocklist guardrail — the canonical "5-line safety gate" demo.
- An `after_tool_callback` PII redactor that strips sensitive fields from a tool's output before the model sees them.
- A `before_tool_callback` that swaps a real API call for a mock — useful for tests.

**Running cost:** under $0.01.

# Setup

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/google/gemini-2.5-flash-lite"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/google/gemini-2.5-flash-lite


## Imports

In [3]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.models.llm_response import LlmResponse
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The Return-to-Override Pattern

This is the mental model for every callback. Each hook has a signature like:

```python
def my_callback(context, request_or_response_or_args):
    # Observe, log, validate — whatever.
    if <some condition>:
        return <a replacement value>   # ← ADK uses this; real call skipped
    return None                        # ← real call proceeds
```

`None` means "carry on." A returned object means "use this instead of doing the thing."

That's the entire callback API. Simple; powerful. Guardrails, caches, mocks, redactors — all the same pattern, different return values.

# Common Helpers

In [4]:
APP = "m07_callbacks"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, label: str = ""):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}USER: {prompt}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    tag = "[FINAL]" if ev.is_final_response() else "[step]"
                    print(f"{prefix}{tag} {ev.author}: {p.text.strip()[:200]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"{prefix}[tool_call] {p.function_call.name}({args})")
                if p.function_response:
                    print(f"{prefix}[tool_resp] {p.function_response.response}")
    print()

print("✅ chat() ready.")

✅ chat() ready.


# Demo 1 — `before_model_callback`: The Blocklist Guardrail

Runs just before every LLM call. Sees the full request ADK is about to send — user message, system instruction, chat history, tool definitions.

Return `None` to let the call proceed. Return an `LlmResponse` to **skip the LLM entirely** and use your canned response instead. Zero tokens billed when you short-circuit.

In [5]:
BLOCKED_WORDS = ["password", "secret", "credit card"]

def blocklist_guardrail(callback_context, llm_request):
    """Short-circuits the LLM if the latest user message contains a blocked word."""
    # llm_request.contents is the chat history, latest turn last.
    latest_user = ""
    if llm_request.contents:
        for part in llm_request.contents[-1].parts or []:
            if part.text:
                latest_user = part.text.lower()
                break

    for bad in BLOCKED_WORDS:
        if bad in latest_user:
            # Return a canned response. ADK uses this; the LLM is not called.
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(text=(
                        f"I can't help with requests involving '{bad}'. "
                        f"Please rephrase your question."
                    ))],
                )
            )
    return None  # No block; proceed normally.

guarded_agent = LlmAgent(
    name="guarded_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="A helpful agent with a blocklist guardrail.",
    instruction="You are helpful and concise. Answer in one short paragraph.",
    before_model_callback=blocklist_guardrail,
)

await chat(guarded_agent, "What is photosynthesis?", "normal")
await chat(guarded_agent, "What's the CEO's password?", "blocked")

[normal] USER: What is photosynthesis?


[normal] [FINAL] guarded_agent: Photosynthesis is the process used by plants, algae, and cyanobacteria to convert light energy into chemical energy, which is stored in organic compounds. This process involves taking in carbon dioxid

[blocked] USER: What's the CEO's password?
[blocked] [FINAL] guarded_agent: I can't help with requests involving 'password'. Please rephrase your question.



Two runs. The first passed — the callback saw no blocked words, returned `None`, the LLM ran, normal answer. The second hit the blocklist — the callback returned a canned `LlmResponse`, ADK used it, **the LLM was never called**.

You will not see a `[step]` event for LLM deliberation on the blocked run. The final response comes straight from the callback.

The pattern scales: swap the `in` check for regex, a PII classifier, a toxicity model, or anything else that can decide yes-or-no in Python. The returned `LlmResponse` can be whatever you want — a refusal, a redirect, a templated answer. **The code-level check is a wall.** An instruction is a polite request; this is enforcement.

# Demo 2 — `after_tool_callback`: PII Redaction

Runs just after a tool returns, before the result goes back to the model. Sees the tool, the arguments it was called with, and the return value.

Return `None` to send the tool's actual return to the model. Return a replacement value to have the model see **that** instead. The tool still ran — you can't un-run side effects — but you can filter what the model learns about the results.

The use case below: an HR lookup tool returns fields the model shouldn't see. Strip them in the callback.

In [6]:
# A fake HR database — includes sensitive fields.
def lookup_employee(name: str) -> dict:
    """Look up an employee by name."""
    DB = {
        "Alice": {"name": "Alice", "email": "alice@company.com",
                  "department": "Engineering",
                  "salary": "72000 EUR", "ssn": "881234567",
                  "home_address": "Main St 42, Bratislava"},
        "Bob":   {"name": "Bob",   "email": "bob@company.com",
                  "department": "Marketing",
                  "salary": "65000 EUR", "ssn": "921112233",
                  "home_address": "Oak Ave 8, Kosice"},
    }
    return DB.get(name, {"error": f"No employee named {name}."})

SENSITIVE_FIELDS = {"salary", "ssn", "home_address", "date_of_birth"}

def redact_pii(tool, args, tool_context, tool_response):
    """Replace sensitive fields with [REDACTED] before the model sees them."""
    if isinstance(tool_response, dict):
        cleaned = dict(tool_response)
        for key in SENSITIVE_FIELDS & cleaned.keys():
            cleaned[key] = "[REDACTED]"
        return cleaned
    return None  # No change

hr_agent = LlmAgent(
    name="hr_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Answers employee-info questions with PII redaction.",
    instruction="Use lookup_employee to answer. Report what you can see.",
    tools=[lookup_employee],
    after_tool_callback=redact_pii,
)

await chat(hr_agent, "Look up Alice's contact info and tell me what you know about her.")

USER: Look up Alice's contact info and tell me what you know about her.


[tool_call] lookup_employee({'name': 'Alice'})
[tool_resp] {'name': 'Alice', 'email': 'alice@company.com', 'department': 'Engineering', 'salary': '[REDACTED]', 'ssn': '[REDACTED]', 'home_address': '[REDACTED]'}


[FINAL] hr_agent: I know Alice works in the Engineering department and her email is alice@company.com. I cannot share her home address, salary, or social security number.



Look at the `[tool_resp]` event in the output. Salary, SSN, home address — all `[REDACTED]` by the time they reach the model.

The tool function itself returned the full record. Your Python code is still aware of the sensitive fields — useful for audit logs, database writes, or downstream systems that legitimately need them. But the model's context contains only what it needs to answer the user's question.

This pattern scales to any "filter outputs on the way out" need: truncate overly long results, redact keys, rename fields the model gets confused by, add audit tags.

# Demo 3 — `before_tool_callback`: Mocking for Tests

Runs just before a tool executes. Sees the tool and the args the model wants to call it with.

Return `None` to let the tool run. Return a dict to **skip the tool** and pretend it returned your dict instead. Useful for testing (no real API hit), for short-circuiting an expensive call when a cheap cache check says "already answered," or for per-environment behavior (return canned data in dev, real data in prod).

In [7]:
# A real API tool — expensive, external, we don't want it firing in tests.
def fetch_stock_price(ticker: str) -> dict:
    """Fetch the current stock price for a ticker."""
    # In production this would hit a real finance API.
    # Here we just return a plausible-looking fake to prove the callback is short-circuiting.
    print(f"   ↪ REAL fetch_stock_price called for {ticker}")
    return {"ticker": ticker, "price": 123.45, "currency": "USD", "source": "live"}

MOCK_RESPONSES = {
    "AAPL": {"ticker": "AAPL", "price": 180.00, "currency": "USD", "source": "mock"},
    "GOOG": {"ticker": "GOOG", "price": 140.00, "currency": "USD", "source": "mock"},
}

def mock_in_tests(tool, args, tool_context):
    """Return a mock response instead of hitting the real API."""
    if tool.name == "fetch_stock_price":
        ticker = args.get("ticker", "").upper()
        if ticker in MOCK_RESPONSES:
            print(f"   ↪ Short-circuiting fetch_stock_price for {ticker} with mock.")
            return MOCK_RESPONSES[ticker]
    return None  # Let the real tool run.

stock_agent = LlmAgent(
    name="stock_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports stock prices.",
    instruction=(
        "For every stock-related question, call fetch_stock_price with the ticker "
        "the user mentioned. Never answer without calling it. Report the numeric "
        "price and currency."
    ),
    tools=[fetch_stock_price],
    before_tool_callback=mock_in_tests,
)

await chat(stock_agent, "What's AAPL trading at?")
await chat(stock_agent, "What about MSFT?")

USER: What's AAPL trading at?


[tool_call] fetch_stock_price({'ticker': 'AAPL'})
   ↪ Short-circuiting fetch_stock_price for AAPL with mock.
[tool_resp] {'ticker': 'AAPL', 'price': 180.0, 'currency': 'USD', 'source': 'mock'}


[FINAL] stock_agent: Apple is trading at $180.

USER: What about MSFT?


[tool_call] fetch_stock_price({'ticker': 'MSFT'})
   ↪ REAL fetch_stock_price called for MSFT
[tool_resp] {'ticker': 'MSFT', 'price': 123.45, 'currency': 'USD', 'source': 'live'}


[FINAL] stock_agent: MSFT is currently trading at $123.45 USD.



The first run asked for AAPL — which has a mock entry. The callback short-circuited with the mock; the `REAL` print didn't fire.

The second run asked for MSFT — which is NOT in `MOCK_RESPONSES`. The callback returned `None`, so the real tool ran (and you see `↪ REAL` in the output).

This pattern is how you make agent code **unit-testable**: inject mocks via `before_tool_callback` for the tests, leave the callback off in production, and the same agent code runs both paths without modification.

# The Six Hooks — Reference

You've seen the three most-used hooks. Here are all six with one-line descriptions of what each is good for:

| Hook | Fires | Common use |
|---|---|---|
| `before_agent_callback(ctx)` | Before the agent processes input | Pre-flight state setup; reject inputs on session-level conditions |
| `after_agent_callback(ctx)` | After the agent finishes | Final-output logging; write summary state |
| `before_model_callback(ctx, req)` | Before every LLM call | **Guardrails**, caching, prompt injection checks |
| `after_model_callback(ctx, resp)` | After every LLM call | Response filtering; telemetry; cost tracking |
| `before_tool_callback(tool, args, ctx)` | Before every tool executes | **Mocking**, cache lookups, argument validation |
| `after_tool_callback(tool, args, ctx, resp)` | After every tool returns | **PII redaction**, result caching, error classification |

And two lesser-used error hooks: `on_model_error_callback` and `on_tool_error_callback` for custom error recovery (default: the error propagates).

All six use the same return-to-override pattern. All six are plain Python functions you pass in at agent construction.

# Callbacks vs Alternatives — When to Pick Which

Callbacks are not the only way to intervene in an agent's lifecycle. ADK offers several mechanisms; each has its place.

| Mechanism | Scope | When |
|---|---|---|
| **Instruction prompt** | One agent | Soft preferences, style, default behavior |
| **Tool function code** | One tool | Guards on irreversible operations (M02) |
| **Callback** | One agent's lifecycle | Cross-cutting concerns — guardrails, PII, caching |
| **Plugin** (newer, recommended for app-wide) | Whole runner, all agents | Org-wide policies; audit logging |

The rule of thumb: **callbacks for agent-specific logic, plugins for app-wide policy.** A blocklist that applies to one specialist agent → callback. A company-wide audit trail that must fire on every agent in the app → plugin.

Plugins (`google.adk.plugins`) are out of scope for this module — M10 touches on them for production deployments.

# The Observability Gotcha

Worth naming so you're not surprised. **Callback execution does not automatically appear in the traces that ADK emits via OpenTelemetry** — verified through the 2.7 release (this might change; check release notes). You will see `invoke_agent`, `call_llm`, and tool spans; no callback spans.

Concretely: if you're using Cloud Trace, Langfuse, or Arize to inspect runs, you will see the LLM calls, the tool calls, and the state deltas. You will **not** see "before_model_callback ran, returned None" as a span. If you need to observe callback execution in traces, add `print` statements or explicit telemetry inside the callback.

This is a known gap — documented in Google's own ADK blog posts. Production guidance: instrument your callbacks manually if you rely on them for policy decisions.

# Your Turn

1. **A caching `before_model_callback`.** Implement a cache keyed on the last user message. On a cache hit, return a canned `LlmResponse` with the cached answer. On a miss, return `None` (proceed); save the final response via `output_key=` or an `after_model_callback`. Verify cache hits don't call the LLM.
2. **A length-limiting `after_model_callback`.** If the model's response is longer than 200 characters, return a replacement `LlmResponse` containing only the first 200 characters with "..." appended. Test on a "write me a long essay" prompt.
3. **A logging `before_tool_callback`.** Log every tool invocation to a list in session state: `tool_context.state["temp:tool_log"]`. After the run, inspect the list to see the audit trail.
4. **An error-path test.** Make `fetch_stock_price` raise an exception. Add an `on_tool_error_callback` that catches the exception and returns a friendly `{"error": "Stock service unavailable"}` dict. Verify the agent still produces a graceful response.

# Key Takeaways

- **Six lifecycle hooks:** before/after × agent/model/tool. Plus two error hooks.
- **Return-to-override:** return `None` to proceed, return an object to short-circuit with your value.
- **`before_model_callback`** is the guardrail hook — blocklist, PII, prompt-injection check. The LLM is not called when you short-circuit.
- **`after_tool_callback`** is the output-filter hook — PII redaction, truncation, re-shaping.
- **`before_tool_callback`** is the mocking/cache hook — return a value instead of running the tool.
- **Callbacks for agent-specific logic; Plugins for app-wide policy.**
- **Observability gap:** callbacks don't automatically appear in OpenTelemetry traces. Instrument manually if you rely on them.

# Next up — M08: Memory

Session state covered short-term. M08 is about the other thing an agent needs to remember — long-term facts that survive across every session, for weeks or months. We swap the in-memory session service for `DatabaseSessionService` backed by SQLite, and introduce the `load_memory` tool and `MemoryService` for explicit long-term recall. Plus a revisit of the Skeptical Memory pattern, now with teeth.